# 模块一：全球引子数据清洗与提取 (背景铺垫)
**执行人**：成员4 (数据工程师)
**核心目标**：证明中国在过去20年取得了健康成就（预期寿命提升），但随着老龄化，以心血管疾病为首的“慢性病”抬头，医疗资源面临极大挑战。
**数据源**：
1. `WDICSV.csv` (世界银行宏观经济与预期寿命)
2. `全球各国核心疾病与死亡估算数据` (23个年份的文件夹，提取四大慢性病致死趋势)
**处理逻辑**：
- 经济数据：精准过滤 CHN, USA, IND，提取 GDP 与预期寿命，并将1960-2024的宽表逆透视为长表。
- 疾病数据：批量遍历23个CSV文件，清洗尾部版权脏数据，精准提取“四大慢性病”及老龄化代表疾病，进行时间序列拼接。

In [3]:
import pandas as pd
import os
import glob
import warnings
warnings.filterwarnings('ignore') # 保持控制台清爽

print("开始处理第一部分：WDI 宏观经济与预期寿命数据...")

# 1. 精确的绝对路径
wdi_path = r'D:\2026_BigData_Project\data\raw\WDICSV.csv' 
df_wdi = pd.read_csv(wdi_path)

# 2. 筛选对比国家与核心指标
target_countries = ['CHN', 'USA', 'IND']
# 提取GDP和预期寿命（证明经济发展与寿命延长的成就）
target_indicators = ['GDP (current US$)', 'Life expectancy at birth, total (years)']

df_wdi_filtered = df_wdi[
    (df_wdi['Country Code'].isin(target_countries)) & 
    (df_wdi['Indicator Name'].isin(target_indicators))
]

# 3. 宽表转长表 (Melt)
year_columns = [str(year) for year in range(1960, 2025)] 
existing_years = [col for col in year_columns if col in df_wdi_filtered.columns]

df_wdi_long = pd.melt(
    df_wdi_filtered,
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=existing_years,
    var_name='Year',
    value_name='Value'
)

# 4. 清理空值与格式转换
df_wdi_long = df_wdi_long.dropna(subset=['Value'])
df_wdi_long['Year'] = df_wdi_long['Year'].astype(int)

# 5. 导出经济小表
out_wdi_path = r'D:\2026_BigData_Project\data\processed\Global_WDI_filtered.csv'
os.makedirs(os.path.dirname(out_wdi_path), exist_ok=True)
df_wdi_long.to_csv(out_wdi_path, index=False)

print(f"✅ WDI数据处理完成！已导出至：{out_wdi_path}")
print(df_wdi_long.head(3))

开始处理第一部分：WDI 宏观经济与预期寿命数据...
✅ WDI数据处理完成！已导出至：D:\2026_BigData_Project\data\processed\Global_WDI_filtered.csv
  Country Name Country Code                           Indicator Name  \
0        China          CHN                        GDP (current US$)   
1        China          CHN  Life expectancy at birth, total (years)   
2        India          IND                        GDP (current US$)   

   Indicator Code  Year         Value  
0  NY.GDP.MKTP.CD  1960  5.984624e+10  
1  SP.DYN.LE00.IN  1960  3.341900e+01  
2  NY.GDP.MKTP.CD  1960  3.702988e+10  


In [4]:
print("\n开始处理第二部分：20年全球疾病时间序列数据批量提取...")

# 1. 疾病文件夹绝对路径
folder_path = r'D:\2026_BigData_Project\data\raw\全球各国核心疾病与死亡估算数据'
all_files = glob.glob(os.path.join(folder_path, "*.csv"))
print(f"侦测到目标文件夹下共有 {len(all_files)} 个 CSV 文件。")

df_list = []

# 2. 严谨的业务筛选条件（完全基于你的截图提取）
target_locations = ['中国', '美利坚合众国', '印度']
target_causes = [
    '所有原因',         # 用来算占比的分母基数
    '心血管疾病',       # 慢性病致死 No.1
    '肿瘤',             # 极度消耗重症资源的代表（根据你的截图是'肿瘤'）
    '慢性呼吸系统疾病', # 老年人高发
    '糖尿病和肾病',     # 消耗基层/门诊资源最大的“不死癌症”
    '神经系统疾病'      # 包含老年痴呆，完美扣题老龄化
] 

# 3. 开启批处理遍历
for file in all_files:
    df_temp = pd.read_csv(file)
    
    # 【核心清洗】切掉表尾的 IHME 英文版权说明（防止脏数据污染）
    df_temp = df_temp.dropna(subset=['地理位置', '死亡或受伤原因'])
    
    # 精准交叉筛选
    df_filtered = df_temp[
        (df_temp['地理位置'].isin(target_locations)) & 
        (df_temp['死亡或受伤原因'].isin(target_causes))
    ]
    
    if not df_filtered.empty:
        df_list.append(df_filtered)

# 4. 垂直拼装完整面板数据
if len(df_list) > 0:
    df_disease_final = pd.concat(df_list, ignore_index=True)
    
    # 5. 导出疾病大表
    out_disease_path = r'D:\2026_BigData_Project\data\processed\Global_Disease_filtered_AllYears.csv'
    df_disease_final.to_csv(out_disease_path, index=False)
    print(f"✅ 疾病数据批量提取合并完成！总计 {df_disease_final.shape[0]} 行。")
    print(f"✅ 已导出至：{out_disease_path}")
    print(df_disease_final.head(3))
else:
    print("❌ 提取失败，大筐里是空的，请检查筛选字段。")


开始处理第二部分：20年全球疾病时间序列数据批量提取...
侦测到目标文件夹下共有 24 个 CSV 文件。
✅ 疾病数据批量提取合并完成！总计 720 行。
✅ 已导出至：D:\2026_BigData_Project\data\processed\Global_Disease_filtered_AllYears.csv
  Population    地理位置      年份  年龄  性别 死亡或受伤原因    测量             数值  \
0        全人口  美利坚合众国  2000.0  全部  合计      肿瘤  死亡排名       2.000000   
1        全人口  美利坚合众国  2000.0  全部  合计      肿瘤    死亡  609833.468288   
2        全人口      印度  2000.0  全部  合计      肿瘤  死亡排名       7.000000   

              下限             上限  
0            NaN            NaN  
1  569171.249741  631510.433238  
2            NaN            NaN  


In [5]:
print("================ 交付前数据终极体检 ================\n")

# 1. 体检 WDI 宏观经济数据
print("【1. WDI 宏观经济数据体检报告】")
# 检查空值
print("各列缺失值统计：")
print(df_wdi_long.isnull().sum())
# 检查时间跨度
min_year_wdi = df_wdi_long['Year'].min()
max_year_wdi = df_wdi_long['Year'].max()
print(f"时间跨度：从 {min_year_wdi} 年 到 {max_year_wdi} 年")
print("-" * 40)

# 2. 体检 疾病死亡数据
print("\n【2. 全球疾病死亡数据体检报告】")
# 检查空值
print("各列缺失值统计：")
print(df_disease_final.isnull().sum())
# 检查包含的具体年份 (注意：如果官方表里的年份列名叫别的，比如'Year'，这里会报错，你需要改一下中括号里的字)
if '年份' in df_disease_final.columns:
    years_included = sorted(df_disease_final['年份'].unique())
    print(f"共包含 {len(years_included)} 个年份的数据。")
    print(f"具体年份列表：{years_included}")
elif 'year' in df_disease_final.columns:
    years_included = sorted(df_disease_final['year'].unique())
    print(f"共包含 {len(years_included)} 个年份的数据。")
    print(f"具体年份列表：{years_included}")
else:
    print("提示：未找到名为'年份'的列，请自行核对表头。")

print("\n================ 体检完成 ================")

================ 交付前数据终极体检 ================

【1. WDI 宏观经济数据体检报告】
各列缺失值统计：
Country Name      0
Country Code      0
Indicator Name    0
Indicator Code    0
Year              0
Value             0
dtype: int64
时间跨度：从 1960 年 到 2024 年
----------------------------------------

【2. 全球疾病死亡数据体检报告】
各列缺失值统计：
Population      0
地理位置            0
年份              0
年龄              0
性别              0
死亡或受伤原因         0
测量              0
数值              0
下限            360
上限            360
dtype: int64
共包含 24 个年份的数据。
具体年份列表：[2000.0, 2001.0, 2002.0, 2003.0, 2004.0, 2005.0, 2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0]

================ 体检完成 ================
